## 1. Import Libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Read CSV Files

In [2]:
retrenchment_by_industry = pd.read_csv("Data/retrenchment_by_industry.csv")

## 3. Show First 5 Rows Of Dataset

In [3]:
retrenchment_by_industry.head()

,year,industry,incidence_of_retrenchment
0,1998,manufacturing,62.9
1,1998,"food, beverages and tobacco",15.4
2,1998,textile and wearing apparel,31.1
3,1998,paper products and publishing,27
4,1998,petroleum and chemical products,30.1


## 4. Data Cleaning

#### 4.A. Code to make a copy of retrenchment_by_industry dataframe for cleaning

In [4]:
retrenchment_by_industry_clean = retrenchment_by_industry.copy()
print("Shape:", retrenchment_by_industry_clean.shape)

Shape: (1154, 3)


### 4.B. Dataset Summary before cleaning

In [5]:
column_summary = pd.DataFrame({
    "data_type": retrenchment_by_industry_clean.dtypes.astype(str),
    "missing_count": (retrenchment_by_industry_clean.isna().sum()),
    "missing_percentage": (retrenchment_by_industry_clean.isna().mean() * 100).round(2),
    "unique_count": (retrenchment_by_industry_clean.nunique(dropna = False)
    )
})

display(column_summary)

print("Empty Rows:", retrenchment_by_industry_clean.isna().all(axis=1).sum())
print("Exact duplicate rows:", retrenchment_by_industry_clean.duplicated().sum())
print("Duplicate identifier records:", retrenchment_by_industry_clean.duplicated(subset=["year", "industry"]).sum())

,data_type,missing_count,missing_percentage,unique_count
year,int64,0,0.0,28
industry,str,0,0.0,68
incidence_of_retrenchment,str,0,0.0,301


Empty Rows: 0
Exact duplicate rows: 0
Duplicate identifier records: 0


### 4.C. Checking for Placeholder Values

In [6]:
placeholder_values = {
    "",
    "na",
    "n/a",
    "null",
    "none",
    "unknown",
    "-"
    }

placeholder_report = []

for column in [
    "industry",
    "incidence_of_retrenchment"
]:
    normalised_values = (
        retrenchment_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    count = normalised_values.isin(placeholder_values).sum()

    placeholder_report.append({
        "Column": column,
        "Placeholder Count": count
    })

display(pd.DataFrame(placeholder_report))

,Column,Placeholder Count
0,industry,0
1,incidence_of_retrenchment,19


### 4.D. Convert Placeholder Values to NaN

In [7]:
for column in [
    "industry",
    "incidence_of_retrenchment"
]:
    normalised_values = (
        retrenchment_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    placeholder_mask = normalised_values.isin(placeholder_values)

    retrenchment_by_industry_clean.loc[placeholder_mask, column] = np.nan


print("Missing values after replacing placeholders:")
display(retrenchment_by_industry_clean.isna().sum())

Missing values after replacing placeholders:


year                          0
industry                      0
incidence_of_retrenchment    19
dtype: int64

### 4.E. Check for Duplicate Records

In [8]:
duplicate_rows = retrenchment_by_industry_clean.duplicated().sum()
print("Duplicate rows found:", duplicate_rows)

Duplicate rows found: 0


### 4.F. Removing Duplicate Records (if any)

In [9]:
rows_before = len(retrenchment_by_industry_clean)
retrenchment_by_industry_clean = (
    retrenchment_by_industry_clean.drop_duplicates()
)

### resetting the index after dropping duplicates to ensure a clean DataFrame
retrenchment_by_industry_clean.reset_index(drop=True, inplace=True)

rows_removed = (
    rows_before - len(retrenchment_by_industry_clean)
)

print("Rows removed:", rows_removed)

Rows removed: 0


### 4.G. Validating the Year Column then Finding and Removing Invalid Years

In [10]:
### check whether every year is numeric
year_numeric_test = pd.to_numeric(
    retrenchment_by_industry_clean["year"],
    errors="coerce"
)

### finding invalid years
invalid_year_mask = (
    year_numeric_test.isna() & retrenchment_by_industry_clean["year"].notna()
)

print("Rows containing invalid year values:", invalid_year_mask.sum())

display(
    retrenchment_by_industry_clean.loc[
        invalid_year_mask
    ]
)

### Removing Invalid Years
rows_before = len(retrenchment_by_industry_clean)

retrenchment_by_industry_clean = (retrenchment_by_industry_clean.loc[
    ~invalid_year_mask
        ].copy())

rows_removed = (rows_before - len(retrenchment_by_industry_clean))

print("Rows removed due to invalid years:", rows_removed)

Rows containing invalid year values: 0


,year,industry,incidence_of_retrenchment


Rows removed due to invalid years: 0


### 4.H. Converting Year to Integer and Checking Year Distribution

In [11]:
### Converting Year to Integer

retrenchment_by_industry_clean["year"] = (
    pd.to_numeric(retrenchment_by_industry_clean["year"],
        errors = "coerce").astype("Int64")
)

print(retrenchment_by_industry_clean["year"].dtype)


### Check year Distribution

display(
    retrenchment_by_industry_clean["year"].value_counts(dropna=False).sort_index()
)

Int64


year
1998    28
1999    28
2000    43
2001    43
2002    43
2003    43
2004    43
2005    43
2006    42
2007    42
2008    42
2009    42
2010    42
2011    42
2012    42
2013    42
2014    42
2015    42
2016    42
2017    42
2018    42
2019    42
2020    42
2021    42
2022    42
2023    42
2024    42
2025    42
Name: count, dtype: Int64

### 4.I. Checking Consistency for Industry Categories

In [12]:
display(retrenchment_by_industry_clean["industry"].value_counts(dropna=False))

industry
manufacturing                            28
food, beverages and tobacco              28
transport equipment                      28
other manufacturing industries           28
construction                             28
                                         ..
petroleum and chemical products           2
transport, storage and communications     2
financial intermediation                  2
insurance and pension funding             2
business and real estate services         2
Name: count, Length: 68, dtype: int64

### 4.J. Standardising Text Columns

In [13]:
text_columns = [
    "industry"
]

for column in text_columns:

    retrenchment_by_industry_clean[column] = (
        retrenchment_by_industry_clean[column]
        .astype("string")
        .str.strip()
        .str.title()
    )

### 4.K. Verifying Standardised Categories

In [14]:
print("Unique values in Industry:")
display(sorted(retrenchment_by_industry_clean["industry"].dropna().unique()))

Unique values in Industry:


['Accommodation',
 'Accommodation And Food Services',
 'Administrative And Support Services',
 'Air Transport And Supporting Services',
 'Architectural And Engineering Services',
 'Arts, Entertainment And Recreation',
 'Broadcasting And Publishing',
 'Business And Real Estate Services',
 'Cleaning And Landscaping',
 'Community, Social And Personal Services',
 'Construction',
 'Education',
 'Electrical Products',
 'Electronic Products',
 'Electronic, Computer And Optical Products',
 'Fabricated Metal Products',
 'Fabricated Metal Products, Machinery And Equipment',
 'Financial And Insurance Services',
 'Financial Institutions',
 'Financial Intermediation',
 'Financial Services',
 'Food And Beverage Services',
 'Food, Beverages And Tobacco',
 'Health And Social Services',
 'Health, Social And Community Services',
 'Hotels',
 'Hotels And Restaurants',
 'Information And Communications',
 'Insurance',
 'Insurance And Pension Funding',
 'Insurance Services',
 'It And Other Information Servic

### 4.L. Converting Incidence of Retrenchment to Numeric and Validating Values

In [15]:
retrenchment_by_industry_clean["incidence_of_retrenchment"] = pd.to_numeric(
    retrenchment_by_industry_clean["incidence_of_retrenchment"],
    errors="coerce"
)

print(retrenchment_by_industry_clean["incidence_of_retrenchment"].dtype)

negative_values = (
    retrenchment_by_industry_clean["incidence_of_retrenchment"] < 0
).sum()

print("Negative incidence of retrenchment values:", negative_values)

float64
Negative incidence of retrenchment values: 0


_The 19 missing values in incidence_of_retrenchment came from the '-' placeholders in the raw data (industries with no retrenchment recorded for that year). These are kept as NaN rather than dropped or filled with 0, since the two mean different things: NaN means the figure was not reported, while 0 would mean no retrenchment happened. Whether to drop or impute them will be decided later based on how they affect the research question._

### 4.M. Summary of data after cleaning

In [16]:
print("Final Shape:")
print(retrenchment_by_industry_clean.shape)

print("\nMissing Values:")
display(retrenchment_by_industry_clean.isna().sum())

print("\nDuplicate Rows:")
print(retrenchment_by_industry_clean.duplicated().sum())

print("\nData Types:")
display(retrenchment_by_industry_clean.dtypes)

Final Shape:
(1154, 3)

Missing Values:


year                          0
industry                      0
incidence_of_retrenchment    19
dtype: int64


Duplicate Rows:
0

Data Types:


year                           Int64
industry                      string
incidence_of_retrenchment    float64
dtype: object